In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "processed_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

PROJECT_REGION = "RAIPUR"
FACILITY_TYPES = ["School", "Hospital"]

INPUT_FILES = {
    "gwl_2021_2025": BASE_DIR / "GWL_2021-2025_Telementry_Hourly.xlsx",
    "gwl_2026_2030": BASE_DIR / "GWL_2026-2030_Telementry_Hourly.csv",
    "rainfall_2021_2025": BASE_DIR / "rainfall_2021_2025_2026-03-13.csv",
    "rainfall_2026_2030": BASE_DIR / "rainfall_2026_2030_2026-03-13.csv",
    "river_2021_2025_raipur": BASE_DIR / "Riverwater_level_2021_2025_raipur.csv",
    "river_2026_2030_raipur": BASE_DIR / "riverwater_level_2026_2030_raipur.csv",
}

def normalize_token(name):
    return "".join(ch for ch in str(name).lower() if ch.isalnum())

def pick_exact_or_best(columns, exact_token, include_tokens, exclude_tokens=None):
    exclude_tokens = exclude_tokens or []
    normalized = {col: normalize_token(col) for col in columns}

    for col, token in normalized.items():
        if token == exact_token:
            return col

    for col, token in normalized.items():
        if all(t in token for t in include_tokens) and not any(e in token for e in exclude_tokens):
            return col
    return None

def pick_timestamp_col(columns):
    normalized = {col: normalize_token(col) for col in columns}
    for preferred in ["dataacquisitiontime", "timestamp", "datetime", "date"]:
        for col, token in normalized.items():
            if token == preferred or preferred in token:
                return col
    return None

def pick_value_col(columns, metric):
    normalized = {col: normalize_token(col) for col in columns}
    if metric == "gwl":
        preferred = ["groundwaterlevel", "gwl"]
    elif metric == "rainfall":
        preferred = ["rainfall", "precip"]
    else:
        preferred = ["riverwaterlevel", "waterlevel", "level"]

    for key in preferred:
        for col, token in normalized.items():
            if key in token:
                return col
    return None

def standardize_environment(df, metric, source_file):
    district_col = pick_exact_or_best(
        df.columns,
        exact_token="district",
        include_tokens=["district"],
        exclude_tokens=["lgd", "code"],
    )
    station_col = pick_exact_or_best(
        df.columns,
        exact_token="station",
        include_tokens=["station"],
        exclude_tokens=["code"],
    )
    state_col = pick_exact_or_best(
        df.columns,
        exact_token="state",
        include_tokens=["state"],
        exclude_tokens=["lgd", "code"],
    )
    lat_col = pick_exact_or_best(df.columns, exact_token="latitude", include_tokens=["lat"])
    lon_col = pick_exact_or_best(df.columns, exact_token="longitude", include_tokens=["lon"])
    ts_col = pick_timestamp_col(df.columns)
    value_col = pick_value_col(df.columns, metric)

    if district_col is None or ts_col is None or value_col is None:
        raise ValueError(
            f"Missing required columns for {metric} in {source_file.name}: "
            f"district={district_col}, timestamp={ts_col}, value={value_col}"
        )

    out = pd.DataFrame({
        "metric": metric,
        "station": df[station_col].astype(str).str.strip() if station_col else "UNKNOWN",
        "district": df[district_col].astype(str).str.strip().str.upper(),
        "state": df[state_col].astype(str).str.strip() if state_col else np.nan,
        "latitude": pd.to_numeric(df[lat_col], errors="coerce") if lat_col else np.nan,
        "longitude": pd.to_numeric(df[lon_col], errors="coerce") if lon_col else np.nan,
        "timestamp": pd.to_datetime(df[ts_col], errors="coerce", dayfirst=True),
        "value": pd.to_numeric(df[value_col], errors="coerce"),
        "source_file": source_file.name,
    })

    out = out.dropna(subset=["timestamp", "value"])
    out = out[out["district"] == PROJECT_REGION].copy()
    out = out.sort_values(["timestamp", "station"], kind="stable").reset_index(drop=True)
    return out

def read_source(path):
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)

datasets = []
for key, path in INPUT_FILES.items():
    if not path.exists():
        print(f"Skipped missing file: {path.name}")
        continue

    if "gwl" in key:
        metric = "gwl"
    elif "rainfall" in key:
        metric = "rainfall"
    else:
        metric = "river_level"

    try:
        raw_df = read_source(path)
        clean_df = standardize_environment(raw_df, metric=metric, source_file=path)
        datasets.append(clean_df)
        print(f"Loaded {path.name}: {len(clean_df):,} {PROJECT_REGION} rows")
    except Exception as ex:
        print(f"Failed {path.name}: {ex}")

if not datasets:
    raise RuntimeError("No usable source datasets were loaded.")

environment_long = pd.concat(datasets, ignore_index=True)
environment_long = environment_long.drop_duplicates(subset=["metric", "station", "timestamp", "value"])

env_long_path = OUTPUT_DIR / "environmental_long_raipur.csv"
environment_long.to_csv(env_long_path, index=False)

environment_daily = (
    environment_long.assign(date=environment_long["timestamp"].dt.date)
    .groupby(["metric", "date"], as_index=False)
    .agg(
        mean_value=("value", "mean"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        records=("value", "count"),
    )
)
env_daily_path = OUTPUT_DIR / "environmental_daily_raipur.csv"
environment_daily.to_csv(env_daily_path, index=False)

export_path = BASE_DIR / "export.csv"
facilities = pd.DataFrame()
if export_path.exists():
    raw_fac = pd.read_csv(export_path)

    raw_fac["longitude"] = pd.to_numeric(raw_fac.get("X"), errors="coerce")
    raw_fac["latitude"] = pd.to_numeric(raw_fac.get("Y"), errors="coerce")

    district_series = raw_fac.get("addr:district", "").astype(str).str.upper()
    city_series = raw_fac.get("addr:city", "").astype(str).str.upper()
    raw_fac["district_norm"] = district_series.where(district_series.str.len() > 0, city_series)

    amenity = raw_fac.get("amenity", "").astype(str).str.lower()
    education = raw_fac.get("education", "").astype(str).str.lower()
    healthcare = raw_fac.get("healthcare", "").astype(str).str.lower()

    is_school = amenity.eq("school") | education.str.contains("school", na=False)
    is_hospital = amenity.eq("hospital") | healthcare.str.contains("hospital", na=False)

    raw_fac["facility_type"] = np.select(
        [is_school, is_hospital],
        ["School", "Hospital"],
        default="",
    )

    raw_fac = raw_fac[
        raw_fac["district_norm"].str.contains(PROJECT_REGION, na=False)
        & raw_fac["facility_type"].isin(FACILITY_TYPES)
    ].copy()

    raw_fac["facility_name"] = raw_fac.get("name", "").astype(str).str.strip()
    raw_fac.loc[raw_fac["facility_name"].eq(""), "facility_name"] = (
        raw_fac["facility_type"] + "_Unknown"
    )

    raw_fac["facility_id"] = raw_fac.get("@id", raw_fac.get("id", "")).astype(str).str.strip()
    raw_fac.loc[raw_fac["facility_id"].eq(""), "facility_id"] = (
        "GEN_" + raw_fac.index.astype(str)
    )

    facilities = raw_fac[[
        "facility_id",
        "facility_name",
        "facility_type",
        "district_norm",
        "latitude",
        "longitude",
    ]].rename(columns={"district_norm": "district"})

    facilities = facilities.dropna(subset=["latitude", "longitude"])
    facilities = facilities.drop_duplicates(subset=["facility_id"]).reset_index(drop=True)

if facilities.empty:
    station_points = (
        environment_long[["station", "district", "latitude", "longitude"]]
        .dropna(subset=["latitude", "longitude"])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    if station_points.empty:
        station_points = pd.DataFrame(
            [{"station": "RAIPUR_CENTRAL", "district": PROJECT_REGION, "latitude": 21.2514, "longitude": 81.6296}]
        )

    rng = np.random.default_rng(42)
    facility_rows = []
    for i, row in station_points.head(25).iterrows():
        for facility_type in FACILITY_TYPES:
            facility_rows.append(
                {
                    "facility_id": f"F{i+1:03d}_{facility_type[0]}",
                    "facility_name": f"{facility_type}_{i+1:03d}",
                    "facility_type": facility_type,
                    "district": PROJECT_REGION,
                    "latitude": float(row["latitude"]) + float(rng.normal(0, 0.01)),
                    "longitude": float(row["longitude"]) + float(rng.normal(0, 0.01)),
                }
            )
    facilities = pd.DataFrame(facility_rows)

facilities["district"] = facilities["district"].astype(str).str.upper()
facilities = facilities[
    facilities["district"].str.contains(PROJECT_REGION, na=False)
    & facilities["facility_type"].isin(FACILITY_TYPES)
].copy()

facilities_path = OUTPUT_DIR / "facilities_master_generated_raipur.csv"
facilities.to_csv(facilities_path, index=False)

def nearest_metric_value(metric_name, fac_lat, fac_lon):
    sub = environment_long[environment_long["metric"] == metric_name].copy()
    if sub.empty:
        return np.nan
    sub = sub.sort_values("timestamp").groupby("station", as_index=False).tail(1)
    sub = sub.dropna(subset=["latitude", "longitude"])
    if sub.empty:
        return np.nan
    distances = ((sub["latitude"] - fac_lat) ** 2 + (sub["longitude"] - fac_lon) ** 2) ** 0.5
    idx = distances.idxmin()
    return float(sub.loc[idx, "value"])

features = facilities.copy()
features["gwl_value"] = features.apply(lambda r: nearest_metric_value("gwl", r["latitude"], r["longitude"]), axis=1)
features["rainfall_value"] = features.apply(lambda r: nearest_metric_value("rainfall", r["latitude"], r["longitude"]), axis=1)
features["river_level_value"] = features.apply(lambda r: nearest_metric_value("river_level", r["latitude"], r["longitude"]), axis=1)

outage_files = [
    BASE_DIR / "combined_outages.csv",
    BASE_DIR / "live_outages.csv",
    BASE_DIR / "all_outages_today.csv",
    BASE_DIR / "all_outages_tomorrow.csv",
]
outage_rows = 0
for p in outage_files:
    if p.exists():
        try:
            outage_rows += len(pd.read_csv(p))
        except Exception:
            pass

outage_score = min(1.0, outage_rows / 1000.0) if outage_rows > 0 else 0.5
features["outage_risk"] = outage_score

def minmax(series):
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series([0.5] * len(s), index=s.index)
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series([0.5] * len(s), index=s.index)
    return (s - lo) / (hi - lo)

features["water_availability"] = minmax(features["gwl_value"]).fillna(0.5)
features["flood_safety"] = (1 - minmax(features["river_level_value"])).fillna(0.5)
features["rainfall_stability"] = (1 - minmax(features["rainfall_value"])).fillna(0.5)
features["electricity_reliability"] = 1 - features["outage_risk"]

features["crs"] = features[[
    "water_availability",
    "flood_safety",
    "rainfall_stability",
    "electricity_reliability",
]].mean(axis=1)

features["risk_level"] = pd.cut(
    features["crs"],
    bins=[-0.01, 0.40, 0.70, 1.01],
    labels=["High", "Medium", "Low"],
)
features["last_updated"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

features = features[
    (features["district"].str.contains(PROJECT_REGION, na=False))
    & (features["facility_type"].isin(FACILITY_TYPES))
].copy()

features_path = OUTPUT_DIR / "facility_features_raipur.csv"
features.to_csv(features_path, index=False)

scoreboard = features[[
    "facility_id",
    "facility_name",
    "facility_type",
    "district",
    "crs",
    "risk_level",
    "last_updated",
]].sort_values("crs")
scoreboard_path = OUTPUT_DIR / "facility_crs_raipur.csv"
scoreboard.to_csv(scoreboard_path, index=False)

prediction_path = OUTPUT_DIR / "crs_predictions_raipur_facilities.csv"
try:
    scoreboard.to_csv(prediction_path, index=False)
except PermissionError:
    prediction_path = OUTPUT_DIR / f"crs_predictions_raipur_facilities_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
    scoreboard.to_csv(prediction_path, index=False)
    print("Prediction file was locked; wrote to fallback file:", prediction_path.name)

print("\nCreated files:")
for p in [env_long_path, env_daily_path, facilities_path, features_path, scoreboard_path, prediction_path]:
    print("-", p.name, f"({p.stat().st_size:,} bytes)")

print("\nProject scope check:")
print("- Region:", PROJECT_REGION)
print("- Facility types:", ", ".join(FACILITY_TYPES))
print("- Facilities used for CRS:", len(facilities))
print("- Rows in prediction file:", len(scoreboard))

print("\nTop 10 priority facilities (lowest CRS):")
print(scoreboard.head(10).to_string(index=False))

Loaded GWL_2021-2025_Telementry_Hourly.xlsx: 8,746 RAIPUR rows
Loaded GWL_2026-2030_Telementry_Hourly.csv: 0 RAIPUR rows
Loaded rainfall_2021_2025_2026-03-13.csv: 16,861 RAIPUR rows
Loaded rainfall_2026_2030_2026-03-13.csv: 7 RAIPUR rows
Loaded Riverwater_level_2021_2025_raipur.csv: 189,990 RAIPUR rows
Loaded riverwater_level_2026_2030_raipur.csv: 7,379 RAIPUR rows

Created files:
- environmental_long_raipur.csv (28,258,109 bytes)
- environmental_daily_raipur.csv (153,612 bytes)
- facilities_master_generated_raipur.csv (12,300 bytes)
- facility_features_raipur.csv (29,531 bytes)
- facility_crs_raipur.csv (15,964 bytes)
- crs_predictions_raipur_facilities.csv (15,964 bytes)

Project scope check:
- Region: RAIPUR
- Facility types: School, Hospital
- Facilities used for CRS: 148
- Rows in prediction file: 148

Top 10 priority facilities (lowest CRS):
     facility_id                                   facility_name facility_type district      crs risk_level        last_updated
 node/695662